# Graph Neural Network Training

This notebook trains a Linear-Time GNN (LTGNN) for venue recommendations.

Features:
- Heterogeneous graph with user, venue, topic, and region nodes
- Fixed-point iteration (avoids over-smoothing)
- EVR sampling for scalability
- gBCE loss to reduce overconfidence

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

import torch

from src.config import get_config, PROCESSED_DATA_DIR, MODEL_DIR, CHECKPOINT_DIR
from src.models.gnn import (
    HeteroGraphBuilder,
    LTGNN,
    GNNTrainer,
    TrainTestSplit,
    gBCELoss,
)
from src.utils.helpers import set_seed, get_device

# Setup
set_seed(42)
config = get_config()
device = get_device('cuda')  # or 'mps', 'cpu'
print(f"Using device: {device}")

## 1. Load Data and Embeddings

In [ ]:
# Load interaction data
train_reviews = pd.read_parquet(PROCESSED_DATA_DIR / 'train_reviews.parquet')
val_reviews = pd.read_parquet(PROCESSED_DATA_DIR / 'val_reviews.parquet')

print(f"Train interactions: {len(train_reviews)}")
print(f"Val interactions: {len(val_reviews)}")
print(f"Unique users: {train_reviews['user_id'].nunique()}")
print(f"Unique venues: {train_reviews['business_id'].nunique()}")

In [ ]:
# Load user MBTI profiles (from Phase 2)
user_profiles_path = PROCESSED_DATA_DIR / 'user_mbti_profiles.parquet'
user_emb_path = PROCESSED_DATA_DIR / 'user_mbti_embeddings.npy'

if user_profiles_path.exists():
    user_profiles = pd.read_parquet(user_profiles_path)
    print(f"Loaded {len(user_profiles)} user profiles")
else:
    # Create from train data
    user_profiles = pd.DataFrame({'user_id': train_reviews['user_id'].unique()})
    print(f"Created {len(user_profiles)} user profiles from train data")

if user_emb_path.exists():
    user_embeddings = np.load(user_emb_path)
    print(f"User embeddings shape: {user_embeddings.shape}")
else:
    # Random initialization
    EMBED_DIM = 64
    user_embeddings = np.random.randn(len(user_profiles), EMBED_DIM).astype(np.float32) * 0.1
    print(f"Created random user embeddings: {user_embeddings.shape}")

In [ ]:
# Load venue topics and embeddings (from Phase 3)
venue_topics_path = MODEL_DIR / 'bertopic' / 'venue_topics.parquet'
venue_emb_path = MODEL_DIR / 'bertopic' / 'venue_embeddings.npy'

if venue_topics_path.exists():
    venue_topics = pd.read_parquet(venue_topics_path)
    print(f"Loaded {len(venue_topics)} venue topics")
else:
    # Create from train data
    venue_topics = pd.DataFrame({
        'venue_id': train_reviews['business_id'].unique(),
        'topic': 0
    })
    print(f"Created {len(venue_topics)} venue records from train data")

if venue_emb_path.exists():
    venue_embeddings = np.load(venue_emb_path)
    print(f"Venue embeddings shape: {venue_embeddings.shape}")
else:
    # Random initialization
    EMBED_DIM = 64
    venue_embeddings = np.random.randn(len(venue_topics), EMBED_DIM).astype(np.float32) * 0.1
    print(f"Created random venue embeddings: {venue_embeddings.shape}")

## 2. Build Heterogeneous Graph

In [ ]:
# Initialize graph builder
builder = HeteroGraphBuilder(device=device)

# Add user nodes
builder.add_user_nodes(
    user_profiles['user_id'].tolist(),
    user_embeddings,
)

# Add venue nodes
builder.add_venue_nodes(
    venue_topics['venue_id'].tolist(),
    venue_embeddings,
)

In [ ]:
# Add user-venue edges from training data
builder.add_user_venue_edges(
    train_reviews['user_id'].tolist(),
    train_reviews['business_id'].tolist(),
    train_reviews['stars'].tolist() if 'stars' in train_reviews.columns else None,
)

In [ ]:
# Graph statistics
stats = builder.get_statistics()
print("Graph Statistics:")
print(f"  Node types: {stats['node_types']}")
print(f"  Edge types: {stats['edge_types']}")
print("\nNodes:")
for node_type, info in stats['nodes'].items():
    print(f"  {node_type}: {info['count']} nodes, {info['feature_dim']} features")
print("\nEdges:")
for edge_type, info in stats['edges'].items():
    print(f"  {edge_type}: {info['count']} edges")

In [ ]:
# Get edge indices
train_edge_index = builder.edges[('user', 'visits', 'venue')].edge_index

# Create validation edges
val_user_idx = []
val_venue_idx = []
user_node = builder.nodes['user']
venue_node = builder.nodes['venue']

for _, row in val_reviews.iterrows():
    user_idx = user_node.get_idx(row['user_id'])
    venue_idx = venue_node.get_idx(row['business_id'])
    if user_idx is not None and venue_idx is not None:
        val_user_idx.append(user_idx)
        val_venue_idx.append(venue_idx)

val_edge_index = torch.tensor([val_user_idx, val_venue_idx], dtype=torch.long)

print(f"Train edges: {train_edge_index.size(1)}")
print(f"Val edges: {val_edge_index.size(1)}")

## 3. Create LTGNN Model

In [ ]:
# Model hyperparameters
HIDDEN_DIM = 128
EMBEDDING_DIM = 64
NUM_ITERATIONS = 10
DROPOUT = 0.2

# Create model
model = LTGNN(
    user_input_dim=user_embeddings.shape[1],
    venue_input_dim=venue_embeddings.shape[1],
    hidden_dim=HIDDEN_DIM,
    embedding_dim=EMBEDDING_DIM,
    num_iterations=NUM_ITERATIONS,
    dropout=DROPOUT,
)

# Count parameters
num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model parameters: {num_params:,}")
print(model)

## 4. Train the Model

In [ ]:
# Configure training
config.gnn.num_epochs = 50  # Reduce for demo
config.gnn.batch_size = 1024
config.gnn.learning_rate = 0.001

# Convert to tensors
user_features = torch.tensor(user_embeddings, dtype=torch.float32)
venue_features = torch.tensor(venue_embeddings, dtype=torch.float32)

# Create trainer
trainer = GNNTrainer(
    model=model,
    user_features=user_features,
    venue_features=venue_features,
    train_edge_index=train_edge_index,
    val_edge_index=val_edge_index,
    config=config.gnn,
    device=device,
    use_gbce=True,  # Use gBCE loss
    gbce_t=0.8,     # Temperature
)

In [ ]:
# Train
history = trainer.train()

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Loss
train_loss = [h['loss'] for h in history['train_history']]
axes[0].plot(train_loss, label='Train Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss')
axes[0].legend()
axes[0].grid(True)

# AUC
val_auc = [h['auc'] for h in history['val_history']]
axes[1].plot(val_auc, label='Val AUC', color='orange')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('AUC')
axes[1].set_title('Validation AUC')
axes[1].legend()
axes[1].grid(True)

# NDCG@10
val_ndcg = [h['ndcg_at_10'] for h in history['val_history']]
axes[2].plot(val_ndcg, label='Val NDCG@10', color='green')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('NDCG@10')
axes[2].set_title('Validation NDCG@10')
axes[2].legend()
axes[2].grid(True)

plt.tight_layout()
plt.show()

print(f"\nBest NDCG@10: {history['best_metric']:.4f}")

## 5. Extract Embeddings

In [ ]:
# Get final embeddings
user_emb, venue_emb = trainer.get_embeddings()

print(f"User embeddings: {user_emb.shape}")
print(f"Venue embeddings: {venue_emb.shape}")

In [ ]:
# Visualize embedding space with t-SNE
from sklearn.manifold import TSNE

# Sample for visualization
n_sample = 500
user_sample = user_emb[:n_sample].cpu().numpy()
venue_sample = venue_emb[:n_sample].cpu().numpy()

# Combine and reduce
combined = np.vstack([user_sample, venue_sample])
tsne = TSNE(n_components=2, random_state=42, perplexity=30)
reduced = tsne.fit_transform(combined)

# Plot
plt.figure(figsize=(10, 8))
plt.scatter(
    reduced[:n_sample, 0], reduced[:n_sample, 1],
    c='blue', alpha=0.5, label='Users', s=20
)
plt.scatter(
    reduced[n_sample:, 0], reduced[n_sample:, 1],
    c='red', alpha=0.5, label='Venues', s=20
)
plt.legend()
plt.title('User and Venue Embeddings (t-SNE)')
plt.xlabel('Dimension 1')
plt.ylabel('Dimension 2')
plt.tight_layout()
plt.show()

## 6. Test Recommendations

In [ ]:
# Get recommendations for a sample user
sample_user_idx = 0
sample_user_id = user_profiles.iloc[sample_user_idx]['user_id']

rec_indices, rec_scores = trainer.recommend_for_user(
    user_idx=sample_user_idx,
    k=10,
    exclude_visited=True,
)

print(f"Recommendations for user {sample_user_id}:")
print(f"{'Rank':<6} {'Venue ID':<20} {'Score':<10}")
print("-" * 40)

for rank, (idx, score) in enumerate(zip(rec_indices, rec_scores), 1):
    venue_id = venue_topics.iloc[idx]['venue_id']
    print(f"{rank:<6} {venue_id:<20} {score:.4f}")

In [ ]:
# Check user's actual visits
user_visits = train_reviews[train_reviews['user_id'] == sample_user_id]
print(f"\nUser's actual visits ({len(user_visits)} total):")
print(user_visits[['business_id', 'stars']].head(10))

## 7. Save Results

In [ ]:
# Save embeddings
output_dir = MODEL_DIR / 'gnn'
output_dir.mkdir(parents=True, exist_ok=True)

np.save(output_dir / 'user_gnn_embeddings.npy', user_emb.cpu().numpy())
np.save(output_dir / 'venue_gnn_embeddings.npy', venue_emb.cpu().numpy())

# Save graph
builder.save(output_dir / 'graph')

print(f"Saved embeddings and graph to {output_dir}")

In [ ]:
# Save ID mappings for later use
id_mappings = {
    'user_ids': user_profiles['user_id'].tolist(),
    'venue_ids': venue_topics['venue_id'].tolist(),
}

import json
with open(output_dir / 'id_mappings.json', 'w') as f:
    json.dump(id_mappings, f)

print("Saved ID mappings")

## Summary

We've trained a Linear-Time GNN that:
1. Uses fixed-point iteration (avoids over-smoothing)
2. Employs gBCE loss (reduces overconfidence)
3. Generates user and venue embeddings for recommendations

Next step: **Phase 5** - Combine GNN embeddings with XGBoost for final hybrid recommendations.